## 1) Install

In [58]:
import sys
!{sys.executable} -m pip install groq python-dotenv --quiet

## 2) Imports

In [59]:
import os
import cv2
import time
import json
import torch
import torch.nn as nn
import numpy as np
from PIL import Image
from torchvision import transforms, models
from groq import Groq
from dotenv import load_dotenv
import mediapipe as mp

## 3) Load API Key

In [60]:
load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if GROQ_API_KEY is None:
    raise ValueError("GROQ_API_KEY not found. Make sure .env file exists in project folder.")

print("API key loaded successfully.")

client = Groq(api_key=GROQ_API_KEY)
print("Groq client initialized.")

API key loaded successfully.
Groq client initialized.


## 4) Configuration

In [61]:
# Paths
MODEL_PATH = "best_resnet18_finetuned.pt"
OUT_DIR    = "."

# Classes
letters     = [chr(c) for c in range(ord("A"), ord("Z")+1)]
digits      = [str(d) for d in range(10)]
class_names = letters + digits

# Default demo mode — can be changed live with L/D/B keys
DEMO_MODE      = "both"
active_classes = class_names

# Model settings
IMG_SIZE = 224
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

# LLM settings
LLM_MODEL = "llama-3.1-8b-instant"
print("LLM model:", LLM_MODEL)

# Demo settings
CONFIDENCE_THRESHOLD = 0.50
HOLD_SECONDS         = 1.5
PAUSE_SECONDS        = 15.0
MAX_BUFFER_LENGTH    = 20
FLIP_LIVE_FRAME      = True
SHOW_CROP_WINDOW     = True

print("Config loaded.")
print("Model path exists?", os.path.exists(MODEL_PATH))
print("Default mode: both | Press L=letters D=digits B=both during demo")

DEVICE: cuda
LLM model: llama-3.1-8b-instant
Config loaded.
Model path exists? True
Default mode: both | Press L=letters D=digits B=both during demo


## 5) Load Model

In [62]:
def build_resnet18(num_classes):
    m = models.resnet18(weights=None)
    m.fc = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(m.fc.in_features, num_classes)
    )
    return m

def load_model(path, labels):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing model checkpoint: {path}")

    m = build_resnet18(len(labels))
    m.load_state_dict(torch.load(path, map_location=DEVICE, weights_only=True))
    m = m.to(DEVICE)
    m.eval()
    print(f"Loaded model: {path} ({len(labels)} classes)")
    return m

model_both = load_model(MODEL_PATH, class_names)

Loaded model: runs_asl36_final\best_resnet18_finetuned.pt (36 classes)


## 6) Preprocessing Transform

In [63]:
infer_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def preprocess_hand(crop_bgr):
    rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
    pil = Image.fromarray(rgb)
    t   = infer_tf(pil).unsqueeze(0).to(DEVICE)
    return t

## 7) Predict Function

In [64]:
def predict(crop_bgr):
    rgb    = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
    pil    = Image.fromarray(rgb)
    tensor = infer_tf(pil).unsqueeze(0).to(DEVICE)

    model_for_mode = model_both
    labels_for_mode = active_classes

    with torch.no_grad():
        with torch.amp.autocast("cuda", enabled=(DEVICE == "cuda")):
            logits = model_for_mode(tensor)

    probs = torch.softmax(logits, dim=1)

    active_indices = [class_names.index(c) for c in labels_for_mode]
    active_probs = probs[0][active_indices]
    active_probs   = active_probs / active_probs.sum().clamp_min(1e-8)

    confidence = active_probs.max().item()
    pred_idx   = active_probs.argmax().item()
    pred_class = labels_for_mode[pred_idx]

    topk = min(5, len(labels_for_mode))
    top_vals, top_pos = torch.topk(active_probs, topk)
    top_preds = [
        (labels_for_mode[pos.item()], top_vals[i].item())
        for i, pos in enumerate(top_pos)
    ]

    return pred_class, confidence, top_preds

## 8) LLM Correction Function

In [65]:
COMMON_SHORT_WORDS = {
    "A", "AM", "AN", "AS", "AT", "BE", "BY", "DO", "GO", "HE", "I", "IF",
    "IN", "IS", "IT", "ME", "MY", "NO", "OF", "ON", "OR", "SO", "TO",
    "UP", "US", "WE"
}

def looks_like_acronym_token(token, confidences=None):
    if not token.isalpha() or not token.isupper() or not (2 <= len(token) <= 5):
        return False
    if token in COMMON_SHORT_WORDS:
        return False
    if confidences:
        avg_conf = sum(confidences) / len(confidences)
        return avg_conf >= 0.70
    return True

def llm_correct(char_buffer, confidences):
    if not char_buffer:
        return "", "No characters to correct."

    chars_with_conf = " ".join([
        f"{c}({conf:.2f})"
        for c, conf in zip(char_buffer, confidences)
    ])

    raw_str = "".join(char_buffer)
    if raw_str.isdigit():
        return raw_str, "Numeric token preserved exactly."
    if looks_like_acronym_token(raw_str, confidences):
        return raw_str, "Acronym-like token preserved exactly."

    prompt = f"""You are a post-processing assistant for an ASL (American Sign Language) fingerspelling recognizer.

The classifier produced this sequence of characters with confidence scores:
{chars_with_conf}

Rules you must follow:
1. The input is fingerspelled letter by letter. Treat it as a spelling attempt, not a shorthand.
2. Only correct characters that have LOW confidence (below 0.60) AND where a substitution produces a real English word.
3. Do NOT expand single letters into full words (e.g. do not turn "A" into "and" and "C" into "See" ).
4. Do NOT add or remove characters unless a character is clearly a misfire (confidence below 0.40 and makes no sense in context).
5. If the sequence already spells a real word or name, return it as-is with only capitalization fixed.
6. If you are not confident in a correction, return the raw sequence unchanged.
7. Preserve uncommon words, acronyms, names, and technical-looking tokens exactly when unsure.
8. If you are not very confident, return the word unchanged.
9. Respond ONLY with a JSON object in this exact format with no extra text:
{{"corrected": "word", "explanation": "brief reason"}}


Raw sequence: {raw_str}"""

    try:
        completion = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0,
            max_completion_tokens=200,
            stream=False
        )

        raw    = completion.choices[0].message.content.strip()
        raw    = raw.replace("```json", "").replace("```", "").strip()
        result = json.loads(raw)
        return result.get("corrected", raw_str), result.get("explanation", "")

    except Exception as e:
        return raw_str, f"LLM error: {str(e)}"

def llm_sentence_correct(text):
    if not text.strip():
        return "", "No sentence to correct."

    protected_tokens = [
        tok for tok in text.split()
        if tok.isdigit() or any(ch.isdigit() for ch in tok) or (tok.isupper() and len(tok) >= 6) or looks_like_acronym_token(tok)
    ]
    protected_text = ", ".join(protected_tokens) if protected_tokens else "None"

    prompt = f"""You are a post-processing assistant for ASL fingerspelling output.

Input sentence:
{text}

Rules you must follow:
1. Correct spelling and minor grammar only.
2. Preserve the intended meaning.
3. Do not add new information.
4. Keep the sentence concise and natural.
5. Preserve digits exactly as digits. For example, keep `2` as `2`; do not rewrite it as `to`, `too`, or `two`.
6. Preserve uncommon words, acronyms, names, and technical-looking tokens exactly when unsure.
7. You MUST preserve these protected tokens exactly as written: {protected_text}
8. If you are not very confident, return the sentence unchanged.
9. Respond ONLY with a JSON object in this exact format:
{{"sentence": "corrected sentence", "explanation": "brief reason"}}"""

    try:
        completion = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0,
            max_completion_tokens=200,
            stream=False
        )

        raw = completion.choices[0].message.content.strip()
        raw = raw.replace("```json", "").replace("```", "").strip()
        result = json.loads(raw)
        sentence = result.get("sentence", text)
        if protected_tokens and any(tok not in sentence.split() for tok in protected_tokens):
            return text, "Protected token preservation triggered and kept original sentence."
        return sentence, result.get("explanation", "")

    except Exception as e:
        return text, f"LLM error: {str(e)}"

def process_word_buffer(char_buffer, conf_buffer):
    if not char_buffer:
        return "", "", ""

    raw_str = "".join(char_buffer)
    corrected_word, explanation = llm_correct(char_buffer, conf_buffer)
    return raw_str, corrected_word, explanation

## 9) MediaPipe Hand Detector Setup

In [66]:
mp_hands   = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.7,
    min_tracking_confidence=0.6
)

from collections import deque
box_history = deque(maxlen=1)

def get_hand_crop(frame, hand_landmarks, padding_ratio=0.20, max_frame_fraction=0.60):
    h, w = frame.shape[:2]

    x_coords = [lm.x * w for lm in hand_landmarks.landmark]
    y_coords = [lm.y * h for lm in hand_landmarks.landmark]

    x_min, x_max = int(min(x_coords)), int(max(x_coords))
    y_min, y_max = int(min(y_coords)), int(max(y_coords))

    box_w = x_max - x_min
    box_h = y_max - y_min
    side  = max(box_w, box_h)

    cx = (x_min + x_max) // 2
    cy = (y_min + y_max) // 2

    pad  = int(side * padding_ratio)
    half = side // 2 + pad

    box_history.append((cx, cy, half))

    avg_cx   = int(sum(b[0] for b in box_history) / len(box_history))
    avg_cy   = int(sum(b[1] for b in box_history) / len(box_history))
    avg_half = int(sum(b[2] for b in box_history) / len(box_history))

    x1 = max(0, avg_cx - avg_half)
    y1 = max(0, avg_cy - avg_half)
    x2 = min(w, avg_cx + avg_half)
    y2 = min(h, avg_cy + avg_half)

    if x2 <= x1 or y2 <= y1:
        return None

    crop_area  = (x2 - x1) * (y2 - y1)
    frame_area = h * w
    if crop_area / frame_area > max_frame_fraction:
        return None

    return frame[y1:y2, x1:x2]

print("MediaPipe version:", mp.__version__)
print("MediaPipe hands initialized.")

MediaPipe version: 0.10.9
MediaPipe hands initialized.


## 10) Display Helper

In [67]:
def draw_overlay(frame, pred_class, confidence, char_buffer,
                 confidences, corrected, explanation, sentence_words,
                 final_sentence, sentence_explanation, hand_detected,
                 top_preds):
    h, w = frame.shape[:2]
    overlay = frame.copy()

    cv2.rectangle(overlay, (0, h-280), (w, h), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.6, frame, 0.4, 0, frame)

    color_good = (0, 255, 0)
    color_warn = (0, 165, 255)
    color_info = (255, 255, 255)

    if hand_detected:
        conf_color = color_good if confidence >= CONFIDENCE_THRESHOLD else color_warn
        cv2.putText(frame, f"Predicted: {pred_class}  ({confidence*100:.1f}%)",(10, h-190), cv2.FONT_HERSHEY_SIMPLEX, 0.8, conf_color, 2)
    else:
        cv2.putText(frame, "No hand detected",(10, h-190), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color_warn, 2)

    top_text = "Top-5: " + (" | ".join([f"{c}:{p*100:.1f}%" for c, p in top_preds]) if top_preds else "-")
    cv2.putText(frame, top_text[:90], (10, h-150), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color_info, 2)

    buffer_str = " ".join(char_buffer) if char_buffer else "-"
    cv2.putText(frame, f"Buffer:    {buffer_str}", (10, h-110), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color_info, 2)

    cv2.putText(frame, f"Word:      {corrected if corrected else '-'}",(10, h-75), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color_good, 2)

    sentence_text = final_sentence if final_sentence else " ".join(sentence_words)
    sentence_short = sentence_text[:70] + "..." if len(sentence_text) > 70 else sentence_text
    cv2.putText(frame, f"Sentence:  {sentence_short if sentence_short else '-'}",(10, h-40), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color_good, 1)

    reason_text = sentence_explanation if final_sentence else explanation
    exp_short = reason_text[:60] + "..." if len(reason_text) > 60 else reason_text
    cv2.putText(frame, f"Reason: {exp_short if exp_short else '-'}",(10, h-15), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color_info, 1)

    cv2.putText(frame, f"Mode:{DEMO_MODE} | Q=quit C=clear L=letters D=digits B=both SPACE=word ENTER=sentence",(10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 255), 2)

    return frame

## 11) CER/WER Helpers

In [68]:
def cer(reference, hypothesis):
    ref = list(reference.upper().replace(" ", ""))
    hyp = list(hypothesis.upper().replace(" ", ""))

    if len(ref) == 0:
        return 0.0 if len(hyp) == 0 else 1.0

    d = np.zeros((len(ref)+1, len(hyp)+1))
    for i in range(len(ref)+1):
        d[i][0] = i
    for j in range(len(hyp)+1):
        d[0][j] = j
    for i in range(1, len(ref)+1):
        for j in range(1, len(hyp)+1):
            cost = 0 if ref[i-1] == hyp[j-1] else 1
            d[i][j] = min(d[i-1][j]+1, d[i][j-1]+1, d[i-1][j-1]+cost)

    return d[len(ref)][len(hyp)] / len(ref)

def wer(reference, hypothesis):
    ref = reference.upper().split()
    hyp = hypothesis.upper().split()

    if len(ref) == 0:
        return 0.0 if len(hyp) == 0 else 1.0

    d = np.zeros((len(ref)+1, len(hyp)+1))
    for i in range(len(ref)+1):
        d[i][0] = i
    for j in range(len(hyp)+1):
        d[0][j] = j
    for i in range(1, len(ref)+1):
        for j in range(1, len(hyp)+1):
            cost = 0 if ref[i-1] == hyp[j-1] else 1
            d[i][j] = min(d[i-1][j]+1, d[i][j-1]+1, d[i-1][j-1]+cost)

    return d[len(ref)][len(hyp)] / len(ref)

print("CER/WER helpers loaded.")

CER/WER helpers loaded.


## 12) Main Webcam Loop

In [71]:
# State variables
char_buffer          = []
conf_buffer          = []
running_confs        = []
last_accepted        = None
last_accepted_t      = time.time()
last_global_accept_t = 0.0
last_sign_t          = time.time()
corrected_word       = ""
explanation          = ""
sentence_words       = []
final_sentence       = ""
sentence_explanation = ""
llm_sessions         = []
top_preds            = []
last_valid_crop      = None

box_history.clear()
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    raise RuntimeError("Could not open webcam. Check camera connection.")

print("Webcam opened. Press Q to quit, C to clear, SPACE to commit a word, ENTER to polish the sentence.")

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            print("Failed to read frame.")
            break

        if FLIP_LIVE_FRAME:
            frame = cv2.flip(frame, 1)

        rgb   = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        results       = hands.process(rgb)
        hand_detected = results.multi_hand_landmarks is not None
        pred_class    = "-"
        confidence    = 0.0
        top_preds     = []

        if hand_detected:
            hand_lm = results.multi_hand_landmarks[0]
            
            clean_frame = frame.copy()
            crop = get_hand_crop(clean_frame, hand_lm, padding_ratio=0.30)
            mp_drawing.draw_landmarks(
                frame, hand_lm, mp_hands.HAND_CONNECTIONS
            )

            if crop is not None and crop.size > 0:
                last_valid_crop = crop.copy()
                pred_class, confidence, top_preds = predict(crop)

                if pred_class == last_accepted:
                    running_confs.append(confidence)
                    avg_conf = sum(running_confs) / len(running_confs)

                    if (avg_conf >= CONFIDENCE_THRESHOLD
                            and len(running_confs) >= 3):
                        now = time.time()
                        if (now - last_accepted_t >= HOLD_SECONDS
                                and now - last_global_accept_t >= HOLD_SECONDS):
                            char_buffer.append(pred_class)
                            conf_buffer.append(avg_conf)
                            last_sign_t          = now
                            last_global_accept_t = now
                            running_confs        = []
                            print(f"Accepted: {pred_class} ({avg_conf:.2f})")
                else:
                    last_accepted   = pred_class
                    last_accepted_t = time.time()
                    running_confs   = [confidence]

        # Auto LLM call after pause
        if char_buffer:
            time_since_sign = time.time() - last_sign_t
            if (time_since_sign >= PAUSE_SECONDS or
                    len(char_buffer) >= MAX_BUFFER_LENGTH):
                print(f"Sending to LLM: {''.join(char_buffer)}")
                raw_str, corrected_word, explanation = process_word_buffer(char_buffer, conf_buffer)
                print(f"Corrected: {corrected_word}")
                print(f"Explanation: {explanation}")

                llm_sessions.append({
                    "raw":         raw_str,
                    "corrected":   corrected_word,
                    "explanation": explanation
                })
                if corrected_word:
                    sentence_words.append(corrected_word)
                    final_sentence = ""
                    sentence_explanation = ""
                    print(f"Sentence so far: {' '.join(sentence_words)}")

                char_buffer   = []
                conf_buffer   = []
                running_confs = []
                last_sign_t   = time.time()

        frame = draw_overlay(
            frame, pred_class, confidence,
            char_buffer, conf_buffer,
            corrected_word, explanation,
            sentence_words, final_sentence, sentence_explanation,
            hand_detected, top_preds
        )

        if SHOW_CROP_WINDOW:
            if last_valid_crop is not None:
                crop_view = cv2.resize(last_valid_crop, (224, 224))
            else:
                crop_view = np.zeros((224, 224, 3), dtype=np.uint8)
            cv2.imshow("Hand Crop", crop_view)

        cv2.imshow("ASL Demo", frame)

        key = cv2.waitKey(1) & 0xFF
        if key == ord("q"):
            break
        elif key == ord("c"):
            char_buffer   = []
            conf_buffer   = []
            running_confs = []
            corrected_word = ""
            explanation    = ""
            sentence_words = []
            final_sentence = ""
            sentence_explanation = ""
            print("Buffer cleared.")
        elif key == ord(" "):
            if char_buffer:
                print(f"Committing word from buffer: {''.join(char_buffer)}")
                raw_str, corrected_word, explanation = process_word_buffer(char_buffer, conf_buffer)
                print(f"Corrected: {corrected_word}")
                print(f"Explanation: {explanation}")
                llm_sessions.append({
                    "raw":         raw_str,
                    "corrected":   corrected_word,
                    "explanation": explanation
                })
                if corrected_word:
                    sentence_words.append(corrected_word)
                    final_sentence = ""
                    sentence_explanation = ""
                    print(f"Sentence so far: {' '.join(sentence_words)}")
                char_buffer   = []
                conf_buffer   = []
                running_confs = []
                last_sign_t   = time.time()
        elif key in (10, 13):
            if char_buffer:
                raw_str, corrected_word, explanation = process_word_buffer(char_buffer, conf_buffer)
                print(f"Corrected: {corrected_word}")
                print(f"Explanation: {explanation}")
                llm_sessions.append({
                    "raw":         raw_str,
                    "corrected":   corrected_word,
                    "explanation": explanation
                })
                if corrected_word:
                    sentence_words.append(corrected_word)
                char_buffer   = []
                conf_buffer   = []
                running_confs = []
                last_sign_t   = time.time()
            if sentence_words:
                raw_sentence = ' '.join(sentence_words)
                print(f"Sending sentence to LLM: {raw_sentence}")
                final_sentence, sentence_explanation = llm_sentence_correct(raw_sentence)
                print(f"Final sentence: {final_sentence}")
                print(f"Sentence explanation: {sentence_explanation}")
            else:
                print("No words available for sentence correction.")
        elif key == ord("l"):
            active_classes = letters
            DEMO_MODE      = "letters"
            print("Switched to: Letters only")
        elif key == ord("d"):
            active_classes = digits
            DEMO_MODE      = "digits"
            print("Switched to: Digits only")
        elif key == ord("b"):
            active_classes = class_names
            DEMO_MODE      = "both"
            print("Switched to: Both")

except KeyboardInterrupt:
    print("Interrupted.")
finally:
    cap.release()
    cv2.destroyAllWindows()
    print("Webcam released.")

Webcam opened. Press Q to quit, C to clear, SPACE to commit a word, ENTER to polish the sentence.
Switched to: Letters only
Accepted: A (0.94)
Accepted: P (0.55)
Accepted: P (0.50)
Accepted: L (0.62)
Accepted: E (0.73)
Committing word from buffer: APPLE
Corrected: APPLE
Explanation: No corrections needed, already a valid word
Sentence so far: APPLE
Sending sentence to LLM: APPLE
Final sentence: APPLE
Sentence explanation: Protected token preservation triggered; kept original sentence.
Buffer cleared.
Webcam released.


## 13) Evaluation Summary

In [70]:
if llm_sessions:
    print("="*60)
    print("LLM POST-PROCESSING EVALUATION SUMMARY")
    print("="*60)

    cer_scores = []
    wer_scores = []

    print(f"\n{'Raw':<20} {'Corrected':<20} {'CER':>6} {'WER':>6}")
    print("-" * 55)

    for s in llm_sessions:
        c = cer(s["raw"], s["corrected"])
        w = wer(s["raw"], s["corrected"])
        cer_scores.append(c)
        wer_scores.append(w)
        print(f"{s['raw']:<20} {s['corrected']:<20} {c:>6.3f} {w:>6.3f}")

    print("-" * 55)
    print(f"{'Average':<40} {np.mean(cer_scores):>6.3f} {np.mean(wer_scores):>6.3f}")

    eval_path =  "llm_eval_results.json"
    with open(eval_path, "w") as f:
        json.dump(llm_sessions, f, indent=2)
    print(f"\nResults saved to: {eval_path}")

else:
    print("No LLM sessions recorded yet.")
    print("Run the webcam demo first then come back to this cell.")

LLM POST-PROCESSING EVALUATION SUMMARY

Raw                  Corrected               CER    WER
-------------------------------------------------------
C                    C                     0.000  0.000
CA                   CA                    0.000  0.000
CS                   CS                    0.000  0.000
665                  665                   0.000  0.000
DEEP                 DEEP                  0.000  0.000
LEARMNING            LEARNING              0.111  1.000
-------------------------------------------------------
Average                                   0.019  0.167

Results saved to: runs_asl36_final\llm_eval_results.json
